In [ ]:
!pip install groq numpy scikit-learn mlflow

In [ ]:
import os, json, numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer, util
import mlflow

In [ ]:
os.environ["GROQ_API_KEY"]="gsk_BPcNKZGZHtrEoUwkFsOFWGdyb3FYBkJoqN9yNMbQlGeIU8RNF"
client = Groq(api_key=os.environ["GROQ_API_KEY"])
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
# load evalset
data=[json.loads(l) for l in open('../../data/eval.jsonl')]
queries = [d['query'] for d in data]
gts = [d['ground_truth'] for d in data]

def call_strategy(messages):
    r = client.chat.completions.create(model="llama-3.1-8b-instant", messages=messages)
    return r.choices[0].message.content

# wrapper for three strategies (zero, few-shot, cot)
def gen_zero(query):
    messages=[{"role":"system","content":"You are a helpful resume-job matcher."},
              {"role":"user","content":query}]
    return call_strategy(messages)

def gen_few(query, k_examples):
    # build messages: system + k examples (user+assistant) + user
    # load examples from a local list file (create earlier)
    messages=[{"role":"system","content":"You are a helpful resume-job matcher."}]
    messages += k_examples
    messages.append({"role":"user","content":query})
    return call_strategy(messages)

def gen_cot(query):
    messages=[{"role":"system","content":"You are a careful assistant. Think step-by-step."},
              {"role":"user","content":"Please think step-by-step: " + query}]
    return call_strategy(messages)

# compute embedding-cosine between reply and ground_truth
def score_reply(reply, gt):
    e1 = embedder.encode(reply, convert_to_tensor=True)
    e2 = embedder.encode(gt, convert_to_tensor=True)
    return float(util.cos_sim(e1,e2).item())

# evaluate
results = {"zero": [], "few_k3": [], "few_k5": [], "cot": []}
k3_examples = []  # load your prebuilt k3 example messages
k5_examples = []  # load your prebuilt k5 example messages

for q, gt in zip(queries,gts):
    z = gen_zero(q); results["zero"].append(score_reply(z,gt))
    f3 = gen_few(q,k3_examples); results["few_k3"].append(score_reply(f3,gt))
    f5 = gen_few(q,k5_examples); results["few_k5"].append(score_reply(f5,gt))
    c = gen_cot(q); results["cot"].append(score_reply(c,gt))

# summarize
import numpy as np
for k,v in results.items():
    print(k, np.mean(v))

# MLflow logging (basic)
mlflow.set_experiment("prompt_experiments")
with mlflow.start_run(run_name="compare_strategies"):
    for k,v in results.items():
        mlflow.log_metric(k + "_mean_cosine", float(np.mean(v)))
    # save raw results
    mlflow.log_dict(results, "results.json")